In [9]:
import pandas as pd

In [10]:
# 1. Load the two files: movie -> cast strings, and the finished cast_member table (with IDs)
junction = pd.read_csv("movie_lookup.csv")
cast_member = pd.read_csv("cast_member_extract.csv")

# 2. Repair garbled characters (GÃ©rard -> Gérard); if a name is already fine or can't be repaired, leave it as is
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

# 3. Keep only movieid + cast, drop movies without cast info
cast_member_data = junction[['movieid', 'cast']].dropna().copy()

# 4. Repair encoding, then split "A,B,C" into a list and give each actor their own row
cast_member_data['cast'] = cast_member_data['cast'].apply(fix_encoding)   # safe repair, on the whole string
cast_member_data['cast'] = cast_member_data['cast'].str.split(',')
cast_member_data = cast_member_data.explode('cast')
cast_member_data['cast'] = cast_member_data['cast'].str.strip()

# Remove empty entries and leftover suffix fragments like in cast_member (from names like "Bob Smith, Jr.")
cast_member_data = cast_member_data[~cast_member_data['cast'].isin(['', 'Jr.', 'Sr.', 'II', 'III'])]

# Manual fix for the one name that was cut off in the source data
cast_member_data['cast'] = cast_member_data['cast'].replace({'Zsolt PÃ': 'Zsolt Páll'})

print(cast_member_data)

       movieid             cast
0            1     James Levine
0            1     Steve Martin
1            2    Eiko Masuyama
1            2        Gorô Naya
1            2     Ichirô Nagai
...        ...              ...
11363    11364  Martin Bourdieu
11363    11364     Moisés Arias
11363    11364     Molly Parker
11363    11364      Ryan Barber
11363    11364  Vincent Francia

[48296 rows x 2 columns]


In [11]:
# join the two tables
movie_cast = cast_member_data.merge(
    cast_member,
    left_on= 'cast',
    right_on= 'castname',
    how='left'
)

print(movie_cast['castid'].isna().sum())  # check rows with no ID

0


In [12]:
movie_cast.info

<bound method DataFrame.info of        movieid             cast  castid         castname
0            1     James Levine   10374     James Levine
1            1     Steve Martin   22313     Steve Martin
2            2    Eiko Masuyama    6826    Eiko Masuyama
3            2        Gorô Naya    8703        Gorô Naya
4            2     Ichirô Nagai    9630     Ichirô Nagai
...        ...              ...     ...              ...
48291    11364  Martin Bourdieu   15945  Martin Bourdieu
48292    11364     Moisés Arias   17331     Moisés Arias
48293    11364     Molly Parker   17352     Molly Parker
48294    11364      Ryan Barber   20682      Ryan Barber
48295    11364  Vincent Francia   23975  Vincent Francia

[48296 rows x 4 columns]>

In [13]:
movie_cast.info()
movie_cast.isnull().sum().sum()

movie_cast = movie_cast[['movieid','castid']]

movie_cast = movie_cast.drop_duplicates(
    subset=['movieid', 'castid']
)

movie_cast['castid'] = movie_cast['castid'].astype(int) # change type to int

movie_cast.to_csv("movie_cast.csv",index=False)

<class 'pandas.DataFrame'>
RangeIndex: 48296 entries, 0 to 48295
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movieid   48296 non-null  int64
 1   cast      48296 non-null  str  
 2   castid    48296 non-null  int64
 3   castname  48296 non-null  str  
dtypes: int64(2), str(2)
memory usage: 2.7 MB


In [ ]:
print(movie_cast.shape)

# spot check. Swap the IDs back into names, so we can see whether they point at the right people
check = movie_cast.merge(cast_member, on='castid') # join on castid: adds the castname that belongs to each ID
print(check[check['movieid'] == 2])                # show only movie 2, then compare with its cast list in movie_lookup.csv

(48284, 2)
    movieid  castid              castname
2         2    6826         Eiko Masuyama
3         2    8703             Gorô Naya
4         2    9630          Ichirô Nagai
5         2   13653     Kiyoshi Kobayashi
6         2   13884        Kôhei Miyauchi
7         2   15336           Makio Inoue
8         2   22443        Sumi Shimamoto
9         2   22623  Tadamichi Tsuneizumi
10        2   22781           Tarô Ishida
11        2   24472          Yasuo Yamada
